In [ ]:
!pip install --upgrade gdown
import gdown
folder_url = 'https://drive.google.com/drive/folders/1s5W0Nfz9NEN8gP14tge8GuouUDXs2Ssq'
gdown.download_folder(folder_url, quiet=False, use_cookies=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.2
    Uninstalling gdown-5.2.2:
      Successfully uninstalled gdown-5.2.2


Retrieving folder contents


Processing file 1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF full-shape.zip
Processing file 1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p partial.zip
Processing file 1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM rotate.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF
From (redirected): https://drive.google.com/uc?id=1siZtGusB1LfQVapTvNOiYi8aeKKAgcDF&confirm=t&uuid=ec611d1e-e49e-44cb-9b8d-1f3820aae9b8
To: /content/AffordanceNet/full-shape.zip
100%|██████████| 558M/558M [00:06<00:00, 81.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p
From (redirected): https://drive.google.com/uc?id=1ROzh5JtQ8SiuWelMUtGlX0Ouz3mjim2p&confirm=t&uuid=58126203-f2b6-4f93-88d9-e9a95be1b8c0
To: /content/AffordanceNet/partial.zip
100%|██████████| 2.19G/2.19G [00:26<00:00, 83.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vwFQbYRDQlyxOZk0N0Hlxi8O_bhHxgIM
To: /content/AffordanceNet/rotate.zip
100%|██████████| 1.15M/1.15M [00:00<00:00, 105MB/s]
Download completed


['/content/AffordanceNet/full-shape.zip',
 '/content/AffordanceNet/partial.zip',
 '/content/AffordanceNet/rotate.zip']

In [ ]:
!unzip -o /content/AffordanceNet/full-shape.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/partial.zip -d /content/AffordanceNet/
!unzip -o /content/AffordanceNet/rotate.zip -d /content/AffordanceNet/
!rm /content/AffordanceNet/*.zip

Archive:  /content/AffordanceNet/full-shape.zip
  inflating: /content/AffordanceNet/full_shape_train_data.pkl  
  inflating: /content/AffordanceNet/full_shape_val_data.pkl  
Archive:  /content/AffordanceNet/partial.zip
  inflating: /content/AffordanceNet/partial_train_data.pkl  
  inflating: /content/AffordanceNet/partial_val_data.pkl  
Archive:  /content/AffordanceNet/rotate.zip
  inflating: /content/AffordanceNet/rotate_val_data.pkl  


In [ ]:
# ============================================================
# PyBullet Grasp Evaluation — Full Code
# ============================================================
# Assumes:
#   - You have a trained model checkpoint (e.g., best_xyz3d.pth)
#   - The test_aug list is in memory (from previous cells)
#   - DGCNNAblation class and AFFORDANCE_NAMES are defined
#   - PyBullet and open3d are installed
# ============================================================

!pip install pybullet open3d -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.9 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import open3d as o3d
import pybullet as p
import pybullet_data
from tqdm import tqdm
import pickle

In [ ]:
# ============================================================
# 1. Model definition (same as training)
# ============================================================
def knn(x, k=20):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pw = -xx - inner - xx.transpose(2, 1)
    return pw.topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None: idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx_flat = (idx + idx_base).view(-1)
    x_t = x.transpose(2, 1).contiguous().view(B * N, -1)
    feat = x_t[idx_flat].view(B, N, k, C)
    x_rep = x.transpose(2, 1).view(B, N, 1, C).expand(-1, -1, k, -1)
    return torch.cat([feat - x_rep, x_rep], dim=3).permute(0, 3, 1, 2).contiguous()

class DGCNNAblation(nn.Module):
    def __init__(self, num_classes, input_dim=3, k=20):
        super().__init__()
        self.k = k
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_dim * 2, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.head = nn.Sequential(
            nn.Conv1d(512, 256, 1, bias=False),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(128, num_classes, 1))

    def forward(self, x):
        B, C, N = x.shape
        xyz = x[:, :3, :]
        idx1 = knn(xyz, k=self.k)
        edge = get_graph_feature(x, k=self.k, idx=idx1)
        x1 = self.conv1(edge).max(dim=-1)[0]
        edge = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge).max(dim=-1)[0]
        edge = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge).max(dim=-1)[0]
        x_local = torch.cat([x1, x2, x3], dim=1)
        x_global = x_local.max(dim=-1, keepdim=True)[0].expand(-1, -1, N)
        out = self.head(torch.cat([x_local, x_global], dim=1))
        return out.permute(0, 2, 1)


In [ ]:
import pickle
import numpy as np
from tqdm import tqdm

# ------------------------------------------------------------
# 1. Load original training data (contains full objects)
# ------------------------------------------------------------
with open("/content/AffordanceNet/full_shape_train_data.pkl", "rb") as f:
    train_data = pickle.load(f)

# All affordance names (from first sample)
AFFORDANCE_NAMES = train_data[0]['affordance']
print("All affordances:", AFFORDANCE_NAMES)

# ------------------------------------------------------------
# 2. Select affordances and test categories (same as experiment)
# ------------------------------------------------------------
SELECTED_CLASSES = ['grasp', 'contain', 'wrap_grasp', 'pourable']
SELECTED_INDICES = [AFFORDANCE_NAMES.index(c) for c in SELECTED_CLASSES]

TEST_CATEGORIES = ['Mug', 'Vase', 'Scissors']
all_cats = set(s['semantic class'] for s in train_data)
TEST_CATEGORIES = [c for c in TEST_CATEGORIES if c in all_cats]
print("Test categories:", TEST_CATEGORIES)

# ------------------------------------------------------------
# 3. Filter test samples (only those with at least one positive point for selected affordances)
# ------------------------------------------------------------
def has_positive(sample):
    for idx in SELECTED_INDICES:
        aff = AFFORDANCE_NAMES[idx]
        if (sample['full_shape']['label'][aff].flatten() > 0.5).any():
            return True
    return False

test_split = [s for s in train_data if s['semantic class'] in TEST_CATEGORIES and has_positive(s)]
print(f"Test samples: {len(test_split)}")

# ------------------------------------------------------------
# 4. Build test_aug list (XYZ only, no geometric features)
# ------------------------------------------------------------
test_aug = []
for s in tqdm(test_split, desc="Preparing test samples"):
    coords = s['full_shape']['coordinate'].astype(np.float32)   # (N, 3)
    test_aug.append({
        'features': coords,                # (N, 3) XYZ only
        'labels': s['full_shape']['label'], # dict of raw affordance masks (not binarised, but we don't need them for grasp eval)
        'category': s['semantic class']
    })

# ------------------------------------------------------------
# 5. Save to file
# ------------------------------------------------------------
with open('test_aug.pkl', 'wb') as f:
    pickle.dump(test_aug, f)

print(f"Saved test_aug.pkl with {len(test_aug)} samples.")

In [ ]:
# ============================================================
# 2. Constants
# ============================================================
AFFORDANCE_NAMES = ['grasp', 'contain', 'wrap_grasp', 'pourable']
NUM_CLASSES = len(AFFORDANCE_NAMES)


In [ ]:
# ============================================================
# 3. Load test data (Mug)
# ============================================================
with open('test_aug.pkl', 'rb') as f:
    test_aug = pickle.load(f)

sample = None
for s in test_aug:
    if s['category'] == 'Mug':
        sample = s
        break
assert sample is not None, "No Mug in test set"
coords = sample['features'][:, :3]          # (N,3)


In [ ]:
# ============================================================
# 4. Load model and predict grasp (with fallback)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DGCNNAblation(num_classes=NUM_CLASSES, input_dim=3, k=20).to(device)
model.load_state_dict(torch.load('/content/output/best_xyz3d.pth', map_location=device))
model.eval()

pts_tensor = torch.tensor(coords).unsqueeze(0).permute(0,2,1).to(device)
with torch.no_grad():
    logits = model(pts_tensor)
    grasp_prob = torch.sigmoid(logits[0, :, 0]).cpu().numpy()   # (N,)

# Try a lower threshold first
threshold = 0.1
grasp_idx = np.where(grasp_prob > threshold)[0]
print(f"Grasp points at threshold {threshold}: {len(grasp_idx)}")

# If still zero, fallback to top‑50 most confident points
if len(grasp_idx) == 0:
    print("No points above threshold – using top‑50 most confident grasp predictions.")
    grasp_idx = np.argsort(grasp_prob)[-50:]   # largest probabilities
else:
    # If too many, sample 50
    if len(grasp_idx) > 50:
        grasp_idx = np.random.choice(grasp_idx, size=50, replace=False)

n_candidates = len(grasp_idx)
chosen = grasp_idx
print(f"Using {n_candidates} grasp candidates.")

Grasp points at threshold 0.1: 553
Using 50 grasp candidates.


In [ ]:
# ============================================================
# 5. Convex hull for PyBullet
# ============================================================
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(coords)
hull, _ = pcd.compute_convex_hull()
hull_path = "/content/temp_mug.obj"
o3d.io.write_triangle_mesh(hull_path, hull)


True

In [ ]:
# ============================================================
# 6. PyBullet simulation
# ============================================================
p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")

obj_start_pos = [0, 0, 0.5]
obj_id = p.loadURDF(hull_path, basePosition=obj_start_pos, globalScaling=1.0)
p.changeDynamics(obj_id, -1, lateralFriction=1.0)
for _ in range(100): p.stepSimulation()

# Simple gripper (two fingers)
finger_shape = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.01, 0.01, 0.05])
base_shape   = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
base_id  = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=base_shape, baseVisualShapeIndex=-1)
left_id  = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=finger_shape, baseVisualShapeIndex=-1)
right_id = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=finger_shape, baseVisualShapeIndex=-1)

def execute_grasp(position, normal):
    # Reset object
    p.resetBasePositionAndOrientation(obj_id, obj_start_pos, [0,0,0,1])
    for _ in range(50): p.stepSimulation()

    # Orientation from normal
    approach = -normal / np.linalg.norm(normal)
    if abs(approach[2]) < 0.9:
        perp = np.cross(approach, [0,0,1])
    else:
        perp = np.cross(approach, [1,0,0])
    perp /= np.linalg.norm(perp)
    finger_dir = np.cross(approach, perp)
    finger_dir /= np.linalg.norm(finger_dir)
    rot_mat = np.column_stack((perp, finger_dir, approach))
    #quat = p.getQuaternionFromEuler(p.getEulerFromRotationMatrix(rot_mat))
    quat = p.getQuaternionFromEuler(p.getEulerFromRotationMatrix(rot_mat))

    # Pre‑grasp position
    grasp_center = position + normal * 0.05
    p.resetBasePositionAndOrientation(base_id, grasp_center, quat)

    # Move towards object and close fingers
    for step in range(30):
        new_pos = [grasp_center[i] - 0.002 * normal[i] for i in range(3)]
        p.resetBasePositionAndOrientation(base_id, new_pos, quat)
        lpos = [new_pos[i] + (0.04 - 0.001*step) * finger_dir[i] for i in range(3)]
        rpos = [new_pos[i] - (0.04 - 0.001*step) * finger_dir[i] for i in range(3)]
        p.resetBasePositionAndOrientation(left_id, lpos, [0,0,0,1])
        p.resetBasePositionAndOrientation(right_id, rpos, [0,0,0,1])
        p.stepSimulation()

    # Lift
    for _ in range(100):
        cur_pos, _ = p.getBasePositionAndOrientation(base_id)
        new_pos = [cur_pos[0], cur_pos[1], cur_pos[2] + 0.002]
        p.resetBasePositionAndOrientation(base_id, new_pos, quat)
        p.stepSimulation()

    obj_pos, _ = p.getBasePositionAndOrientation(obj_id)
    return obj_pos[2] > obj_start_pos[2] + 0.1


error: Cannot load URDF file.

In [ ]:
import os
print("File exists:", os.path.exists(hull_path))
print("File size:", os.path.getsize(hull_path))

File exists: True
File size: 21086


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import open3d as o3d
import pybullet as p
import pybullet_data
from tqdm import tqdm
import pickle, os

# ============================================================
# 1. Model definition (same as training)
# ============================================================
def knn(x, k=20):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pw = -xx - inner - xx.transpose(2, 1)
    return pw.topk(k=k, dim=-1)[1]

def get_graph_feature(x, k=20, idx=None):
    B, C, N = x.shape
    if idx is None: idx = knn(x, k=k)
    device = x.device
    idx_base = torch.arange(0, B, device=device).view(-1, 1, 1) * N
    idx_flat = (idx + idx_base).view(-1)
    x_t = x.transpose(2, 1).contiguous().view(B * N, -1)
    feat = x_t[idx_flat].view(B, N, k, C)
    x_rep = x.transpose(2, 1).view(B, N, 1, C).expand(-1, -1, k, -1)
    return torch.cat([feat - x_rep, x_rep], dim=3).permute(0, 3, 1, 2).contiguous()

class DGCNNAblation(nn.Module):
    def __init__(self, num_classes, input_dim=3, k=20):
        super().__init__()
        self.k = k
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_dim * 2, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv2d(128, 64, 1, bias=False),
            nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 128, 1, bias=False),
            nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.head = nn.Sequential(
            nn.Conv1d(512, 256, 1, bias=False),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(256, 128, 1, bias=False),
            nn.BatchNorm1d(128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Conv1d(128, num_classes, 1))

    def forward(self, x):
        B, C, N = x.shape
        xyz = x[:, :3, :]
        idx1 = knn(xyz, k=self.k)
        edge = get_graph_feature(x, k=self.k, idx=idx1)
        x1 = self.conv1(edge).max(dim=-1)[0]
        edge = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(edge).max(dim=-1)[0]
        edge = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(edge).max(dim=-1)[0]
        x_local = torch.cat([x1, x2, x3], dim=1)
        x_global = x_local.max(dim=-1, keepdim=True)[0].expand(-1, -1, N)
        out = self.head(torch.cat([x_local, x_global], dim=1))
        return out.permute(0, 2, 1)

# ============================================================
# 2. Constants
# ============================================================
AFFORDANCE_NAMES = ['grasp', 'contain', 'wrap_grasp', 'pourable']
NUM_CLASSES = len(AFFORDANCE_NAMES)

# ============================================================
# 3. Load test data (Mug)
# ============================================================
with open('test_aug.pkl', 'rb') as f:
    test_aug = pickle.load(f)

sample = None
for s in test_aug:
    if s['category'] == 'Mug':
        sample = s
        break
assert sample is not None, "No Mug in test set"
coords = sample['features'][:, :3]          # (N,3)

# ============================================================
# 4. Load model and predict grasp (with fallback)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DGCNNAblation(num_classes=NUM_CLASSES, input_dim=3, k=20).to(device)
model.load_state_dict(torch.load('/content/output/best_xyz3d.pth', map_location=device))
model.eval()

pts_tensor = torch.tensor(coords).unsqueeze(0).permute(0,2,1).to(device)
with torch.no_grad():
    logits = model(pts_tensor)
    grasp_prob = torch.sigmoid(logits[0, :, 0]).cpu().numpy()   # (N,)

# Try a lower threshold first
threshold = 0.1
grasp_idx = np.where(grasp_prob > threshold)[0]
print(f"Grasp points at threshold {threshold}: {len(grasp_idx)}")

# If still zero, fallback to top‑50 most confident points
if len(grasp_idx) == 0:
    print("No points above threshold – using top‑50 most confident grasp predictions.")
    grasp_idx = np.argsort(grasp_prob)[-50:]   # largest probabilities
else:
    # If too many, sample 50
    if len(grasp_idx) > 50:
        grasp_idx = np.random.choice(grasp_idx, size=50, replace=False)

n_candidates = len(grasp_idx)
chosen = grasp_idx
print(f"Using {n_candidates} grasp candidates.")

# ============================================================
# 5. Create mesh for PyBullet (convex hull or bounding box fallback)
# ============================================================
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(coords)

# Try convex hull
hull, _ = pcd.compute_convex_hull()
if len(hull.vertices) == 0:   # degenerate hull
    bbox = pcd.get_axis_aligned_bounding_box()
    hull = o3d.geometry.TriangleMesh.create_box(width=bbox.get_extent()[0],
                                                height=bbox.get_extent()[1],
                                                depth=bbox.get_extent()[2])
    hull.translate(bbox.get_min_bound())

# Save as OBJ
hull_path = "/content/temp_mug.obj"
o3d.io.write_triangle_mesh(hull_path, hull)

# ============================================================
# 6. PyBullet simulation (load mesh correctly)
# ============================================================
p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")

obj_start_pos = [0, 0, 0.5]

# Create collision shape from the OBJ mesh
mesh_scale = [1.0, 1.0, 1.0]
col_shape_id = p.createCollisionShape(shapeType=p.GEOM_MESH,
                                      fileName=hull_path,
                                      meshScale=mesh_scale)
visual_shape_id = -1   # use collision shape for visualization as well (or create a separate visual shape)
obj_id = p.createMultiBody(baseMass=0.1,
                           baseCollisionShapeIndex=col_shape_id,
                           baseVisualShapeIndex=visual_shape_id,
                           basePosition=obj_start_pos)
p.changeDynamics(obj_id, -1, lateralFriction=1.0)

# Let object settle
for _ in range(100):
    p.stepSimulation()

# Simple gripper (two fingers)
finger_shape = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.01, 0.01, 0.05])
base_shape   = p.createCollisionShape(p.GEOM_BOX, halfExtents=[0.02, 0.02, 0.02])
base_id  = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=base_shape, baseVisualShapeIndex=-1)
left_id  = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=finger_shape, baseVisualShapeIndex=-1)
right_id = p.createMultiBody(baseMass=0.1, baseCollisionShapeIndex=finger_shape, baseVisualShapeIndex=-1)

# ------------------------------------------------------------
# Fix: rotation matrix -> quaternion (no Euler angles)
# ------------------------------------------------------------
def rotmat2quat(R):
    """Convert 3x3 rotation matrix to quaternion (x,y,z,w) for PyBullet."""
    m00, m01, m02 = R[0,0], R[0,1], R[0,2]
    m10, m11, m12 = R[1,0], R[1,1], R[1,2]
    m20, m21, m22 = R[2,0], R[2,1], R[2,2]
    trace = m00 + m11 + m22
    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        w = 0.25 / s
        x = (m21 - m12) * s
        y = (m02 - m20) * s
        z = (m10 - m01) * s
    elif (m00 > m11) and (m00 > m22):
        s = 2.0 * np.sqrt(1.0 + m00 - m11 - m22)
        w = (m21 - m12) / s
        x = 0.25 * s
        y = (m01 + m10) / s
        z = (m02 + m20) / s
    elif (m11 > m22):
        s = 2.0 * np.sqrt(1.0 + m11 - m00 - m22)
        w = (m02 - m20) / s
        x = (m01 + m10) / s
        y = 0.25 * s
        z = (m12 + m21) / s
    else:
        s = 2.0 * np.sqrt(1.0 + m22 - m00 - m11)
        w = (m10 - m01) / s
        x = (m02 + m20) / s
        y = (m12 + m21) / s
        z = 0.25 * s
    return [x, y, z, w]  # PyBullet quaternion order: x,y,z,w

def execute_grasp(position, normal):
    # Reset object
    p.resetBasePositionAndOrientation(obj_id, obj_start_pos, [0,0,0,1])
    for _ in range(50): p.stepSimulation()

    # Orientation from normal
    approach = -normal / np.linalg.norm(normal)
    if abs(approach[2]) < 0.9:
        perp = np.cross(approach, [0,0,1])
    else:
        perp = np.cross(approach, [1,0,0])
    perp /= np.linalg.norm(perp)
    finger_dir = np.cross(approach, perp)
    finger_dir /= np.linalg.norm(finger_dir)
    rot_mat = np.column_stack((perp, finger_dir, approach))
    #quat = p.getQuaternionFromEuler(p.getEulerFromRotationMatrix(rot_mat))
    quat = rotmat2quat(rot_mat)

    # Pre‑grasp position
    grasp_center = position + normal * 0.05
    p.resetBasePositionAndOrientation(base_id, grasp_center, quat)

    # Move towards object and close fingers
    for step in range(30):
        new_pos = [grasp_center[i] - 0.002 * normal[i] for i in range(3)]
        p.resetBasePositionAndOrientation(base_id, new_pos, quat)
        lpos = [new_pos[i] + (0.04 - 0.001*step) * finger_dir[i] for i in range(3)]
        rpos = [new_pos[i] - (0.04 - 0.001*step) * finger_dir[i] for i in range(3)]
        p.resetBasePositionAndOrientation(left_id, lpos, [0,0,0,1])
        p.resetBasePositionAndOrientation(right_id, rpos, [0,0,0,1])
        p.stepSimulation()

    # Lift
    for _ in range(100):
        cur_pos, _ = p.getBasePositionAndOrientation(base_id)
        new_pos = [cur_pos[0], cur_pos[1], cur_pos[2] + 0.002]
        p.resetBasePositionAndOrientation(base_id, new_pos, quat)
        p.stepSimulation()

    obj_pos, _ = p.getBasePositionAndOrientation(obj_id)
    return obj_pos[2] > obj_start_pos[2] + 0.1

# ============================================================
# 7. Run grasp trials
# ============================================================
successes = 0
centroid = coords.mean(axis=0)
for idx in tqdm(chosen, desc="Grasp trials"):
    pos = coords[idx]
    n = pos - centroid
    n = n / (np.linalg.norm(n) + 1e-8)
    if np.dot(n, pos - centroid) < 0:
        n = -n
    try:
        if execute_grasp(pos, n):
            successes += 1
    except Exception as e:
        print(f"Grasp {idx} error: {e}")

print(f"\nSuccess rate: {successes}/{len(chosen)} = {successes/len(chosen):.2%}")
p.disconnect()

Grasp points at threshold 0.1: 553
Using 50 grasp candidates.


Grasp trials: 100%|██████████| 50/50 [00:01<00:00, 38.67it/s]


Success rate: 22/50 = 44.00%
